# Semana 6-7 — MDP y Backward Induction: Formulación de Bellman y Recursión hacia Atrás

En las semanas 1 a 5 trabajamos cadenas de Markov: sistemas que evolucionan **sin que nadie decida nada**. A partir de esta semana el sistema tiene un agente que **elige una acción en cada estado**, y esa elección cambia tanto la recompensa inmediata como la matriz de transición. Eso es un **Proceso de Decisión de Markov (MDP)**.

Seguimos la misma estructura de las dos sesiones magistrales de la Semana 6:

| Sesión | Tema | Lo que aprenderá |
|:--|:--|:--|
| **Sesión 1** | Formulación del MDP y ecuación de Bellman | La tupla $(S,A,P,r,\gamma)$, una política fija $\pi$, su función de valor $V^\pi$, y cómo evaluarla resolviendo el sistema lineal $(I-\gamma P^\pi)V^\pi = r^\pi$ |
| **Sesión 2** | MDP de horizonte finito y backward induction | La función de valor dependiente del tiempo $V^\pi_t(s)$, el principio de optimalidad de Bellman, y la recursión hacia atrás $V^*_t(s)=\max_a\big[r_t(s,a)+\sum_{s'}P(s'\mid s,a)V^*_{t+1}(s')\big]$, con $V^*_T=g$ |

### Convenciones de programación

El código central utiliza únicamente herramientas trabajadas en semanas anteriores, más una herramienta nueva:

- arreglos de NumPy;
- ciclos `for` y condicionales `if`;
- funciones sencillas;
- `np.max`, `np.argmax`;
- **`np.linalg.solve`**, para resolver sistemas lineales como $(I-\gamma P^\pi)V^\pi=r^\pi$ sin necesidad de iterar;
- **`jmarkov`** (`dtmc`, `dtmdp`, `dtsdp`), que en este cuaderno usamos como la **herramienta de referencia** del curso para plantear y resolver estos modelos.

### En cada ejercicio seguimos siempre el mismo patrón de tres pasos:

1. **Formulamos el modelo y lo resolvemos a mano** con NumPy, desarmando cada pieza de la ecuación de Bellman: la matriz inducida por la política, el vector de recompensas, el sistema lineal o la recursión hacia atrás.

2. **Reproducimos el mismo cálculo con `jmarkov`** (`dtmdp` para horizonte infinito, `dtsdp` para horizonte finito). No lo tratamos como un detalle de relleno: aprenda su sintaxis con cuidado, porque es la herramienta que usará de aquí en adelante (en el Taller 2, en el Quiz 2 y en el proyecto) cada vez que el modelo crezca demasiado para resolverlo a mano cómodamente.
3. **Comparamos los dos resultados.** Si no coinciden, hay un error en alguno de los dos lados.



In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## Herramienta nueva: `np.argmax`

En un MDP necesitamos distinguir siempre entre dos preguntas distintas:

- ¿cuál es el **mejor valor** entre varias acciones? → `np.max`
- ¿cuál es la **posición de la acción** que produce ese valor? → `np.argmax`

```python
valores_por_accion = np.array([4.2, 6.8])

mejor_valor = np.max(valores_por_accion)
indice_mejor_accion = np.argmax(valores_por_accion)
```

`np.max` devuelve `6.8` (el número); `np.argmax` devuelve `1` (la *posición* de ese número dentro del arreglo, es decir, la segunda acción, índice 1).

Durante todo el cuaderno guardaremos las políticas como **índices enteros** (la posición de la acción elegida) y solo al final traduciremos esos índices a nombres legibles. Esta separación entre "el índice que uso para calcular" y "el nombre que uso para reportar" es una buena práctica que evita errores de indexación.

In [2]:
valores_por_accion_ejemplo = np.array([4.2, 6.8])

mejor_valor_ejemplo = np.max(valores_por_accion_ejemplo)
indice_mejor_accion_ejemplo = np.argmax(valores_por_accion_ejemplo)

print("Valores por acción:", valores_por_accion_ejemplo)
print("Mejor valor:", mejor_valor_ejemplo)
print("Índice de la mejor acción:", indice_mejor_accion_ejemplo)

Valores por acción: [4.2 6.8]
Mejor valor: 6.8
Índice de la mejor acción: 1


---
# Parte 1 — Formulación del MDP y ecuación de Bellman de evaluación

## 1.1 La tupla $(S, A, P, r, \gamma)$

Un MDP queda completamente definido por cinco elementos:

| Símbolo | Nombre | Qué es |
|:--|:--|:--|
| $S$ | Espacio de estados | Los mismos estados que ya construíamos para una cadena de Markov |
| $A$ | Espacio de acciones | Las decisiones disponibles en cada estado |
| $P$ | Transición | $P(s' \mid s, a)$: probabilidad de pasar a $s'$ si estoy en $s$ y elijo $a$ — **una matriz de transición por cada acción** |
| $r$ | Recompensa | $r(s,a)$: recompensa (o costo, con signo negativo) inmediata de elegir $a$ en $s$ |
| $\gamma$ | Descuento | $0 \le \gamma < 1$ en horizonte infinito; en horizonte finito puede ser $\gamma=1$ |

La diferencia frente a una cadena de Markov corriente es exactamente esta: antes teníamos **una sola** matriz $P$; ahora tenemos **una matriz $P^a$ por cada acción posible**, y un agente que debe escoger.

## 1.2 Política y función de valor

Una **política** $\pi$ es una regla que asigna una acción a cada estado: $\pi: S \to A$. Por ahora trabajaremos con políticas **fijas** (usted la propone, nosotros la evaluamos); la pregunta de *cuál política es la mejor* pertenece al horizonte infinito con Value Iteration (Semana 7) y a backward induction en horizonte finito (Parte 2 de hoy).

Dada una política fija $\pi$, su **función de valor** $V^\pi(s)$ es la recompensa esperada descontada, empezando en $s$ y siguiendo siempre $\pi$:

$$
V^\pi(s) = \mathbb{E}\Big[\sum_{t=0}^{\infty} \gamma^t\, r(S_t, \pi(S_t)) \;\Big|\; S_0=s\Big].
$$

## 1.3 La ecuación de Bellman de evaluación

Al fijar la política $\pi$, el MDP se reduce a una cadena de Markov corriente: en cada estado $s$ ya no hay elección, la acción es siempre $\pi(s)$. Eso induce:

- una matriz de transición inducida $P^\pi$, cuya fila $s$ es la fila $s$ de la matriz $P^{\pi(s)}$;
- un vector de recompensas inducido $r^\pi$, cuya entrada $s$ es $r(s,\pi(s))$.

La función de valor satisface entonces la **ecuación de Bellman de evaluación**:

$$
V^\pi(s) = r^\pi(s) + \gamma \sum_{s'} P^\pi(s'\mid s)\, V^\pi(s')
\qquad\Longrightarrow\qquad
(I - \gamma P^\pi)\, V^\pi = r^\pi.
$$

Esta es la pieza más importante de la sesión, y conviene remarcar lo que la hace distinta de lo que viene después: **es un sistema lineal**, no hay ningún `max` involucrado, porque la política ya está fija. Eso significa que no hace falta iterar: `np.linalg.solve` la resuelve en un solo paso, exactamente igual a como resolvíamos $\pi = \pi P$ para la distribución estacionaria en las semanas 1-2.

## 1.4 Traducción al lenguaje de los dominios del curso

La misma estructura $(S,A,P,r,\gamma)$ se reinterpreta en cada dominio del curso:

| Dominio | Estado $s$ | Acción $a$ |
|:--|:--|:--|
| Inventarios | Nivel de inventario disponible | Cantidad a pedir |
| Gestión de activos | Condición / edad del activo | Reemplazar o mantener |
| Finanzas | Nivel de riqueza o régimen de mercado | Fracción del portafolio a invertir |
| Logística | Ubicación actual / cola pendiente | Ruta a tomar u orden de atención del servicio |


## 1.5 Ejemplo — Héroes y Villanos: evaluando una política fija

### Contexto

Una ciudad enfrenta ataques periódicos de villanos en dos zonas: el **Centro** y el **Suburbio**. Antes de cada nuevo ataque, el equipo de superhéroes decide en cuál zona posicionarse.

Interpretación del modelo:

- El estado $S_t$ indica la zona donde ocurrió el ataque más reciente.
- La acción $A_t$ indica la zona donde los héroes se posicionan antes del siguiente ataque.
- El estado $S_{t+1}$ indica la zona donde ocurre el siguiente ataque.
- Los héroes ganan la batalla cuando la zona elegida coincide con $S_{t+1}$.

El MDP tiene horizonte infinito y factor de descuento $\gamma=0.9$.

$$
\mathcal{S}=\{\text{Centro},\text{Suburbio}\}, \qquad
\mathcal{A}=\{\text{Ir al Centro},\text{Ir al Suburbio}\}.
$$

Para cada acción se define una matriz de transición (filas = estado actual $S_t$, columnas = estado siguiente $S_{t+1}$):

$$
P^{(\text{Ir al Centro})}
=
\begin{pmatrix}
0.3 & 0.7\\
0.8 & 0.2
\end{pmatrix},
\qquad
P^{(\text{Ir al Suburbio})}
=
\begin{pmatrix}
0.5 & 0.5\\
0.6 & 0.4
\end{pmatrix}.
$$

La recompensa es la probabilidad de que los héroes se encuentren en la misma zona donde ocurre el próximo ataque, es decir $r(s,a)=P(S_{t+1}=\text{zona de } a \mid S_t=s, A_t=a)$. Con filas = estados y columnas = acciones (en el orden Centro, Suburbio / Ir al Centro, Ir al Suburbio):

$$
R=
\begin{pmatrix}
0.3 & 0.5\\
0.8 & 0.4
\end{pmatrix}.
$$

Por ejemplo, si el ataque anterior ocurrió en el Suburbio y los héroes eligen **Ir al Centro**, la probabilidad de ganar la siguiente batalla es $0.8$: esa es la entrada $R[\text{Suburbio}, \text{Ir al Centro}]$.

In [3]:
# --- Espacios S y A -----------------------------------------------
estados_heroes = np.array(["Centro", "Suburbio"])
acciones_heroes = np.array([
    "Ir al Centro",
    "Ir al Suburbio"
])

# --- Una matriz de transicion P^a por cada accion ------------------
# fila = estado actual S_t, columna = estado siguiente S_{t+1}
P_ir_centro = np.array([
    [0.3, 0.7],
    [0.8, 0.2]
])

P_ir_suburbio = np.array([
    [0.5, 0.5],
    [0.6, 0.4]
])

# Guardamos las matrices en una lista indexada como las acciones:
# matrices_heroes_por_accion[indice_accion] -> matriz de esa accion
matrices_heroes_por_accion = [
    P_ir_centro,
    P_ir_suburbio
]

# --- Recompensa r(s, a): filas = estados, columnas = acciones ------
retornos_heroes = np.array([
    [0.3, 0.5],
    [0.8, 0.4]
])

gamma_heroes = 0.9
numero_estados_heroes = len(estados_heroes)
numero_acciones_heroes = len(acciones_heroes)

# Verificacion barata (Sec. 3 del manual de jmarkov): cada P^a debe ser
# estocastica. Antes de seguir, confirmamos que cada fila suma 1.
for indice_accion in range(numero_acciones_heroes):
    sumas_filas = matrices_heroes_por_accion[indice_accion].sum(axis=1)
    print(
        "Accion:", acciones_heroes[indice_accion],
        "-> suma de filas:", np.round(sumas_filas, 6)
    )

Accion: Ir al Centro -> suma de filas: [1. 1.]
Accion: Ir al Suburbio -> suma de filas: [1. 1.]


### Fijamos una política y calculamos $P^\pi$ y $r^\pi$

En vez de buscar la mejor política, **evaluamos** una política concreta que un equipo de héroes conservador podría proponer:

> $\pi(\text{Centro}) = \text{Ir al Centro}$, $\quad\pi(\text{Suburbio}) = \text{Ir al Suburbio}$

es decir, *"quedarse en la zona del último ataque"*. Es una política razonable a primera vista (el villano podría repetir zona), pero **no sabemos todavía si es buena**: eso es justamente lo que la evaluación nos va a decir, y es una pregunta distinta de *cuál es la óptima*, que dejaremos para después.

Formalizamos $\pi$ como un arreglo de índices (la misma convención de índices enteros de la Sección de `np.argmax`) y con él construimos:

- $P^\pi$: en la fila de cada estado $s$, copiamos la fila $s$ de la matriz de la acción $\pi(s)$;
- $r^\pi$: en la entrada de cada estado $s$, copiamos $r(s,\pi(s))$.

In [4]:
# politica_heroes_fija[s] = indice de la accion que la politica elige en el estado s
# indice 0 -> "Ir al Centro", indice 1 -> "Ir al Suburbio"
politica_heroes_fija = np.array([0, 1])  # Centro->Ir al Centro, Suburbio->Ir al Suburbio

P_pi_heroes = np.zeros((numero_estados_heroes, numero_estados_heroes))
r_pi_heroes = np.zeros(numero_estados_heroes)

for estado_actual in range(numero_estados_heroes):

    accion_elegida = politica_heroes_fija[estado_actual]

    # fila s de P^pi = fila s de la matriz de la accion que pi elige en s
    P_pi_heroes[estado_actual, :] = matrices_heroes_por_accion[accion_elegida][estado_actual, :]

    # entrada s de r^pi = r(s, pi(s))
    r_pi_heroes[estado_actual] = retornos_heroes[estado_actual, accion_elegida]

print("Politica evaluada:")
for estado_actual in range(numero_estados_heroes):
    print(
        "  pi(", estados_heroes[estado_actual], ") =",
        acciones_heroes[politica_heroes_fija[estado_actual]]
    )

print()
print("P^pi (matriz inducida):")
print(P_pi_heroes)
print()
print("r^pi (recompensa inducida):", r_pi_heroes)

Politica evaluada:
  pi( Centro ) = Ir al Centro
  pi( Suburbio ) = Ir al Suburbio

P^pi (matriz inducida):
[[0.3 0.7]
 [0.6 0.4]]

r^pi (recompensa inducida): [0.3 0.4]


### Resolvemos $(I-\gamma P^\pi)V^\pi = r^\pi$

Aquí está el paso central de la Sesión 1: resolvemos el sistema lineal **de una sola vez**, tal como hicimos con $\pi(P-I)=0$ para la distribución estacionaria.

In [5]:
identidad_heroes = np.eye(numero_estados_heroes)

# Construimos A = (I - gamma * P^pi) y resolvemos A V = r^pi
A_heroes = identidad_heroes - gamma_heroes * P_pi_heroes

V_pi_heroes = np.linalg.solve(A_heroes, r_pi_heroes)

print("Matriz (I - gamma * P^pi):")
print(A_heroes)
print()
print("V^pi (valor de la politica evaluada):")
for estado_actual in range(numero_estados_heroes):
    print(
        "  V^pi(", estados_heroes[estado_actual], ") =",
        round(V_pi_heroes[estado_actual], 4)
    )

Matriz (I - gamma * P^pi):
[[ 0.73 -0.63]
 [-0.54  0.64]]

V^pi (valor de la politica evaluada):
  V^pi( Centro ) = 3.4961
  V^pi( Suburbio ) = 3.5748


### Verificación por sustitución directa

La comprobación más barata que existe para un MDP resuelto: sustituir $V^\pi$ de vuelta en la ecuación de Bellman de evaluación y confirmar que efectivamente la satisface, es decir que $r^\pi + \gamma P^\pi V^\pi$ reproduce $V^\pi$.

In [6]:
lado_derecho = r_pi_heroes + gamma_heroes * (P_pi_heroes @ V_pi_heroes)

print("V^pi calculado:          ", np.round(V_pi_heroes, 6))
print("r^pi + gamma*P^pi*V^pi:  ", np.round(lado_derecho, 6))
print("¿Coinciden?              ", np.allclose(V_pi_heroes, lado_derecho))

V^pi calculado:           [3.4961 3.5748]
r^pi + gamma*P^pi*V^pi:   [3.4961 3.5748]
¿Coinciden?               True


### Con `jmarkov`

`dtmdp` está pensado para *elegir* entre varias acciones, pero podemos usarlo también para **evaluar** una política fija: basta con construir un MDP degenerado que tenga una única acción, `"seguir_politica"`, cuya matriz de transición sea exactamente $P^\pi$ y cuya recompensa sea exactamente $r^\pi$. Como no hay ninguna otra acción con la cual comparar, `dtmdp.solve()` no tiene margen de elección: el valor que devuelva **tiene que ser** $V^\pi$.

Aprovechamos para fijarnos bien en la sintaxis, porque la volveremos a usar en la Sesión 2 (con `dtsdp`) y en las próximas semanas:

- `dtmdp(states, actions, {accion: matriz}, R, gamma)`: el tercer argumento es un **diccionario** de matrices indexado por el nombre de la acción, y `R` es un arreglo `(n_estados, n_acciones)`.
- `solve(tolerancia)` devuelve dos objetos: el valor `V` **indexado por posición** (igual que en nuestro cálculo a mano) y una política que es un **diccionario por etiqueta** de estado.

In [7]:
from jmarkov.mdp.dtmdp import dtmdp

matrices_heroes_jmarkov = {"seguir_politica": P_pi_heroes}
retornos_heroes_jmarkov = r_pi_heroes.reshape(numero_estados_heroes, 1)  # (n_estados, n_acciones=1)

mdp_evaluacion_heroes = dtmdp(
    estados_heroes,
    np.array(["seguir_politica"]),
    matrices_heroes_jmarkov,
    retornos_heroes_jmarkov,
    gamma_heroes
)

V_pi_jmarkov, politica_jmarkov = mdp_evaluacion_heroes.solve(1e-12)

print("V^pi calculado a mano: ", np.round(V_pi_heroes, 6))
print("V^pi con jmarkov:      ", np.round(np.array(V_pi_jmarkov, dtype=float), 6))
print("¿Coinciden?            ", np.allclose(V_pi_heroes, V_pi_jmarkov))
print()
print("Politica que reporta jmarkov (la unica accion disponible):", politica_jmarkov)

V^pi calculado a mano:  [3.4961 3.5748]
V^pi con jmarkov:       [3.4961 3.5748]
¿Coinciden?             True

Politica que reporta jmarkov (la unica accion disponible): {np.str_('Centro'): np.str_('seguir_politica'), np.str_('Suburbio'): np.str_('seguir_politica')}


> **Cierre de la Sesión 1.** Ya sabemos *evaluar* una política mediante un sistema lineal. Lo que todavía no sabemos es si $\pi=$ "quedarse en la zona del último ataque" es la mejor política posible entre las cuatro políticas deterministas que existen para este MDP (dos estados, dos acciones cada uno). Responder eso exige comparar $V^\pi$ contra el de las demás políticas, o bien resolver la ecuación de Bellman de **optimalidad** (con un `max`, no una igualdad). Ese es exactamente el contenido de la Semana 7 (Value Iteration) y de la Semana 8 (Policy Iteration) del curso, no lo anticipamos aquí.

---
# Sesión 2 — MDP de horizonte finito y backward induction

## 2.1 Teoría

Cuando el problema tiene un **horizonte finito** de $T$ épocas, la política óptima ya no es una sola regla fija: depende también de **cuánto tiempo falta**. No es lo mismo decidir faltando diez semanas que faltando una. Por eso la función de valor gana un subíndice de tiempo: $V_t(s)$.

**Principio de optimalidad de Bellman.** Una política es óptima si, sin importar en qué estado y en qué época nos encontremos, la decisión que toma en ese momento es óptima *dado* que a partir de ahí seguirá siendo óptima. Esto permite resolver el problema completo **hacia atrás**: empezamos en la última época (donde no hay futuro que considerar) y retrocedemos.

**Backward induction.**

$$
V^*_T(s) = g(s) \qquad \text{(valor terminal, sin continuación)}
$$

$$
V^*_t(s) = \max_{a} \left[ r_t(s,a) + \sum_{s'} P(s' \mid s, a)\, V^*_{t+1}(s') \right], \qquad t = T-1, \ldots, 1.
$$

Dos diferencias importantes frente a la Sesión 1:

1. Aquí **sí** hay un `max`: en cada época comparamos todas las acciones y nos quedamos con la mejor, exactamente como hacíamos con `np.argmax`.
2. No hay que iterar hasta converger: el algoritmo converge en **exactamente $T$ pasos**, uno por época, porque cada época solo depende de la siguiente.

En este cuaderno usaremos $g(s) = \max_a r_T(s,a)$ como valor terminal, es decir, en la última época se toma la mejor decisión disponible sin considerar ningún futuro.

## 2.2 `jmarkov` para horizonte finito: `dtsdp`

Así como `dtmdp` resuelve el problema de horizonte infinito, `jmarkov` trae una clase dedicada a horizonte finito: `dtsdp` (*discrete-time stochastic dynamic programming*). La usaremos como herramienta principal en el resto del cuaderno, así que vale la pena entender su interfaz desde ya:

```python
from jmarkov.sdp.dtsdp import dtsdp

proceso = dtsdp(periodos, estados, acciones, matrices_por_periodo, R, gamma)
F, D = proceso.solve(minimize=False)
```

| Argumento | Estructura | Comentario |
|:--|:--|:--|
| `periodos` | `np.array` | Las épocas, por ejemplo `[1, 2, 3]` |
| `matrices_por_periodo` | diccionario **anidado** `{periodo: {accion: matriz}}` | Permite que la dinámica cambie con el tiempo; si no cambia, se repite la misma matriz en cada periodo |
| `R` | arreglo **3D** `(n_periodos, n_estados, n_acciones)` | Igual que `immediate_returns` de `dtmdp`, pero con un eje de tiempo adicional |
| `gamma` | `float` en $[0,1]$ | A diferencia de `dtmdp`, aquí **sí se admite** $\gamma=1$ |

`solve()` devuelve `F` (valor óptimo) y `D` (decisión óptima), **ambas de forma `(n_estados, n_periodos)`**: estados en las filas, periodos en las columnas — la transpuesta de lo que uno esperaría a primera vista. Además, `D` guarda las etiquetas de las acciones truncadas a **un solo carácter** (es una particularidad documentada del paquete), así que conviene usar etiquetas cortas o reconstruir el nombre completo a partir de la primera letra cuando se necesite reportarlo.

Con esa interfaz en mente, resolvemos primero a mano (para entender exactamente qué hace la recursión) y después con `dtsdp`, comparando siempre los dos resultados.

## Ejercicio 2 — MDP de horizonte finito: reemplazo de máquinas

### Contexto:

Una empresa opera una máquina durante tres semanas. Al comienzo de cada semana observa su estado y decide entre **Reemplazar** o **No reemplazar**.

$$
\mathcal S=\{\text{Excelente},\text{Bueno},\text{Promedio},\text{Malo}\}.
$$

Los retornos inmediatos $r(s,a)$ son:

$$
\begin{array}{c|cc}
&\text{Reemplazar}&\text{No reemplazar}\\
\hline
\text{Excelente}&-1000&100\\
\text{Bueno}&-100&80\\
\text{Promedio}&-100&50\\
\text{Malo}&-100&10
\end{array}.
$$

La acción Reemplazar no es factible en Excelente; conservamos una fila de transición válida para no romper la matriz, pero su retorno de $-1000$ garantiza que `argmax` nunca la elija.

Usaremos siempre la convención `V[epoca, estado]` y `politica[epoca, estado]`, coherente con cómo se indexan las matrices de NumPy (primero la fila = época, luego la columna = estado); es la transpuesta de la convención `(estados, periodos)` de `dtsdp`, así que al comparar los dos resultados habrá que trasponer uno de los dos.

In [8]:
epocas_maquina = np.array([1, 2, 3])

estados_maquina = np.array([
    "Excelente", "Bueno", "Promedio", "Malo"
])

acciones_maquina = np.array([
    "Reemplazar", "No reemplazar"
])

gamma_maquina = 0.9

# retornos_maquina[estado, accion] -- filas = estados, columnas = acciones
retornos_maquina = np.array([
    [-1000, 100],
    [-100,   80],
    [-100,   50],
    [-100,   10]
])

# Los retornos y las transiciones son iguales en las tres semanas, por eso
# no necesitamos un arreglo tridimensional para el algoritmo manual: basta
# con reutilizar la misma matriz en cada epoca dentro del ciclo.
P_no_reemplazar = np.array([
    [0.7, 0.3, 0.0, 0.0],
    [0.0, 0.7, 0.3, 0.0],
    [0.0, 0.0, 0.7, 0.3],
    [0.0, 0.0, 0.0, 1.0]
])

P_reemplazar = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.7, 0.3, 0.0, 0.0],
    [0.7, 0.3, 0.0, 0.0],
    [0.7, 0.3, 0.0, 0.0]
])

# indice 0 -> "Reemplazar", indice 1 -> "No reemplazar" (mismo orden de acciones_maquina)
matrices_maquina_por_accion = [P_reemplazar, P_no_reemplazar]

numero_epocas_maquina = len(epocas_maquina)
numero_estados_maquina = len(estados_maquina)
numero_acciones_maquina = len(acciones_maquina)

# Verificacion barata: cada matriz debe ser estocastica por filas.
for indice_accion in range(numero_acciones_maquina):
    print("Accion:", acciones_maquina[indice_accion])
    for estado_actual in range(numero_estados_maquina):
        suma_fila = matrices_maquina_por_accion[indice_accion][estado_actual, :].sum()
        print(" ", estados_maquina[estado_actual], "-> suma:", round(suma_fila, 4))
    print()

Accion: Reemplazar
  Excelente -> suma: 1.0
  Bueno -> suma: 1.0
  Promedio -> suma: 1.0
  Malo -> suma: 1.0

Accion: No reemplazar
  Excelente -> suma: 1.0
  Bueno -> suma: 1.0
  Promedio -> suma: 1.0
  Malo -> suma: 1.0



### Backward induction paso a paso

Traducimos la recursión de la Sección 2.1 directamente a código, en dos partes que corresponden una a una con la teoría:

1. **Época final** ($t=T$, sin continuación): $V_T(s) = \max_a r(s,a)$. En código: para cada estado tomamos el máximo de la fila correspondiente de `retornos_maquina`.
2. **Épocas anteriores** ($t < T$): para cada estado evaluamos las dos acciones, sumando a la recompensa inmediata el valor esperado de continuación $\gamma \sum_{s'} P^a(s,s')\, V_{t+1}(s')$, y guardamos el máximo (`np.max`) y quién lo logra (`np.argmax`).

In [9]:
valor_maquina = np.zeros((numero_epocas_maquina, numero_estados_maquina))
politica_maquina = np.zeros((numero_epocas_maquina, numero_estados_maquina), dtype=int)

ultima_epoca = numero_epocas_maquina - 1

# --- Paso 1: epoca final V_T(s) = max_a r(s,a), sin continuacion --------
for estado_actual in range(numero_estados_maquina):

    valores_por_accion = retornos_maquina[estado_actual, :]

    valor_maquina[ultima_epoca, estado_actual] = np.max(valores_por_accion)
    politica_maquina[ultima_epoca, estado_actual] = np.argmax(valores_por_accion)

# --- Paso 2: recursion hacia atras, de la penultima epoca a la primera --
for epoca in range(numero_epocas_maquina - 2, -1, -1):

    for estado_actual in range(numero_estados_maquina):

        valores_por_accion = np.zeros(numero_acciones_maquina)

        for accion_actual in range(numero_acciones_maquina):

            # continuacion = sum_{s'} P^a(s, s') * V_{t+1}(s')
            continuacion = 0.0
            for estado_siguiente in range(numero_estados_maquina):
                probabilidad = matrices_maquina_por_accion[accion_actual][estado_actual, estado_siguiente]
                continuacion = continuacion + probabilidad * valor_maquina[epoca + 1, estado_siguiente]

            # r_t(s,a) + gamma * continuacion
            valores_por_accion[accion_actual] = retornos_maquina[estado_actual, accion_actual] + gamma_maquina * continuacion

        valor_maquina[epoca, estado_actual] = np.max(valores_por_accion)
        politica_maquina[epoca, estado_actual] = np.argmax(valores_por_accion)

In [10]:
print("Funciones de valor V[epoca, estado]:")
for estado_actual in range(numero_estados_maquina):
    print()
    print("Estado:", estados_maquina[estado_actual])
    for epoca in range(numero_epocas_maquina):
        print("  Semana", epocas_maquina[epoca], "->", round(valor_maquina[epoca, estado_actual], 2))

print()
print("Politica optima:")
for estado_actual in range(numero_estados_maquina):
    print()
    print("Estado:", estados_maquina[estado_actual])
    for epoca in range(numero_epocas_maquina):
        accion_elegida = politica_maquina[epoca, estado_actual]
        print("  Semana", epocas_maquina[epoca], "->", acciones_maquina[accion_elegida])

indice_malo = 3
print()
print("Ganancia esperada desde Malo en la semana 1:", round(valor_maquina[0, indice_malo], 2))

Funciones de valor V[epoca, estado]:

Estado: Excelente
  Semana 1 -> 255.15
  Semana 2 -> 184.6
  Semana 3 -> 100.0

Estado: Bueno
  Semana 1 -> 193.39
  Semana 2 -> 143.9
  Semana 3 -> 80.0

Estado: Promedio
  Semana 1 -> 108.18
  Semana 2 -> 84.2
  Semana 3 -> 50.0

Estado: Malo
  Semana 1 -> 55.15
  Semana 2 -> 19.0
  Semana 3 -> 10.0

Politica optima:

Estado: Excelente
  Semana 1 -> No reemplazar
  Semana 2 -> No reemplazar
  Semana 3 -> No reemplazar

Estado: Bueno
  Semana 1 -> No reemplazar
  Semana 2 -> No reemplazar
  Semana 3 -> No reemplazar

Estado: Promedio
  Semana 1 -> No reemplazar
  Semana 2 -> No reemplazar
  Semana 3 -> No reemplazar

Estado: Malo
  Semana 1 -> Reemplazar
  Semana 2 -> No reemplazar
  Semana 3 -> No reemplazar

Ganancia esperada desde Malo en la semana 1: 55.15


### Con `dtsdp`

Para usar `dtsdp` necesitamos acomodar nuestros datos a su interfaz: un diccionario anidado `{periodo: {accion: matriz}}` (aquí la dinámica no cambia con el tiempo, así que repetimos las mismas dos matrices en cada periodo) y un arreglo 3D de retornos `(periodos, estados, acciones)` (aquí también repetimos la misma matriz de retornos en cada periodo, porque el problema no cambia semana a semana).

Recuerde que `F` que devuelve `dtsdp` tiene forma `(estados, periodos)`, mientras que nuestro `valor_maquina` tiene forma `(epocas, estados)`: son transpuestas una de la otra.

In [11]:
from jmarkov.sdp.dtsdp import dtsdp

# Diccionario anidado periodo -> {accion: matriz}
probabilidades_maquina_jmarkov = {}
for epoca in range(numero_epocas_maquina):
    numero_epoca = epocas_maquina[epoca]
    probabilidades_maquina_jmarkov[numero_epoca] = {
        "Reemplazar": P_reemplazar,
        "No reemplazar": P_no_reemplazar
    }

# Retornos 3D: (periodos, estados, acciones); se repite la misma matriz
retornos_maquina_jmarkov = np.zeros((numero_epocas_maquina, numero_estados_maquina, numero_acciones_maquina))
for epoca in range(numero_epocas_maquina):
    retornos_maquina_jmarkov[epoca, :, :] = retornos_maquina

sdp_maquina = dtsdp(
    epocas_maquina,
    estados_maquina,
    acciones_maquina,
    probabilidades_maquina_jmarkov,
    retornos_maquina_jmarkov,
    gamma_maquina
)

valor_maquina_jmarkov, politica_maquina_jmarkov = sdp_maquina.solve(minimize=False)

print("Valor manual, transpuesto a (estados x epocas) para comparar:")
print(np.round(valor_maquina.T, 3))
print()
print("Valor con dtsdp (estados x epocas):")
print(np.round(np.array(valor_maquina_jmarkov, dtype=float), 3))
print()
print("¿Coinciden?", np.allclose(valor_maquina.T, valor_maquina_jmarkov))

Valor manual, transpuesto a (estados x epocas) para comparar:
[[255.151 184.6   100.   ]
 [193.391 143.9    80.   ]
 [108.176  84.2    50.   ]
 [ 55.151  19.     10.   ]]

Valor con dtsdp (estados x epocas):
[[255.151 184.6   100.   ]
 [193.391 143.9    80.   ]
 [108.176  84.2    50.   ]
 [ 55.151  19.     10.   ]]

¿Coinciden? True


---
## Ejercicio 3 — Newsvendor multiperíodo: política $(s_t,S_t)$ con costos $(K,h,b,c)$

EcoFlor S.A.S. gestiona ramos de flores durante tres semanas. Este es exactamente el ejemplo que el syllabus describe para la Sesión 2 de la Semana 6: *"newsvendor multi-periodo en 4 periodos con costos $(K,h,b,c)$, derivando la política $(s_t,S_t)$ etapa a etapa"* (aquí usamos 3 periodos para mantener las tablas cortas; la lógica es idéntica para 4).

### 3.1 Los cuatro costos del modelo

Antes escribíamos un único "costo de pedido" que en realidad mezclaba dos ideas distintas. Los separamos explícitamente, como pide la notación del curso:

| Símbolo | Nombre | Significado |
|:--|:--|:--|
| $K$ | Costo fijo de ordenar | Se paga **una sola vez** si se pide algo ($a>0$); no depende de cuánto se pida |
| $c$ | Costo variable de ordenar | Se paga **por cada unidad** pedida: costo total de ordenar $= K\cdot\mathbb 1\{a>0\} + c\cdot a$ |
| $h$ | Costo de almacenamiento | Por cada unidad que sobra al final del periodo |
| $b$ | Costo de escasez (*backorder*) | Por cada unidad de demanda que no se puede satisfacer |

$$
\mathcal S=\{0,1,\ldots,5\}\ (\text{inventario}), \qquad \mathcal A=\{0,1,\ldots,5\}\ (\text{unidades a pedir}), \qquad D\in\{0,1,2,3,4\}.
$$

Precio de venta $p=50$. Igual que antes, pedir más de lo que la capacidad permite ($i+a>5$) es infactible y se penaliza fuertemente para que `argmax` nunca lo elija.

In [13]:
capacidad_inventario = 5

precio_venta = 50
K_pedido = 20              # costo fijo de ordenar (K)
c_pedido = 15               # costo variable por unidad pedida (c)
costo_almacenamiento = 8    # costo de almacenamiento (h)
costo_escasez = 15          # costo de escasez / backorder (b)

penalizacion_infactible = -10000

valores_demanda = np.array([0, 1, 2, 3, 4])
probabilidades_demanda = np.array([0.10, 0.30, 0.30, 0.20, 0.10])

epocas_inventario = np.array([1, 2, 3])

estados_inventario = np.array(["0", "1", "2", "3", "4", "5"])

acciones_inventario = np.array(["0", "1", "2", "3", "4", "5"])  # la cantidad pedida, como etiqueta de 1 caracter

numero_epocas_inventario = len(epocas_inventario)
numero_estados_inventario = len(estados_inventario)
numero_acciones_inventario = len(acciones_inventario)

### 3.2 Construcción de retornos con $(K,h,b,c)$

El retorno esperado de pedir $a$ unidades estando en el estado (inventario) $i$ es

$$
r(i,a) = -\Big(K\cdot\mathbb 1\{a>0\} + c\cdot a\Big) + \mathbb{E}_D\Big[p\cdot\min(i+a,D) - h\cdot\max(0,i+a-D) - b\cdot\max(0,D-i-a)\Big].
$$

El primer paréntesis es el costo de ordenar (fijo más variable); dentro de la esperanza están la venta, el costo de almacenamiento del sobrante y el costo de la demanda insatisfecha. La convertimos en código exactamente en ese orden: primero el costo de ordenar (que no depende de la demanda), después la esperanza sobre los cinco valores posibles de $D$.

In [14]:
def construir_retornos_inventario(costo_escasez_modelo):

    retornos = np.zeros((numero_estados_inventario, numero_acciones_inventario))

    for inventario_actual in range(numero_estados_inventario):

        for cantidad_pedida in range(numero_acciones_inventario):

            inventario_disponible = inventario_actual + cantidad_pedida

            if inventario_disponible > capacidad_inventario:

                retornos[inventario_actual, cantidad_pedida] = penalizacion_infactible

            else:

                # Costo de ordenar: K solo si se pide algo, mas c por unidad
                if cantidad_pedida > 0:
                    costo_ordenar = K_pedido + c_pedido * cantidad_pedida
                else:
                    costo_ordenar = 0.0

                retorno_esperado = -costo_ordenar

                for indice_demanda in range(len(valores_demanda)):

                    demanda = valores_demanda[indice_demanda]
                    probabilidad_demanda = probabilidades_demanda[indice_demanda]

                    ventas = min(inventario_disponible, demanda)
                    inventario_sobrante = max(0, inventario_disponible - demanda)
                    demanda_insatisfecha = max(0, demanda - inventario_disponible)

                    retorno_escenario = (
                        precio_venta * ventas
                        - costo_almacenamiento * inventario_sobrante
                        - costo_escasez_modelo * demanda_insatisfecha
                    )

                    retorno_esperado = retorno_esperado + probabilidad_demanda * retorno_escenario

                retornos[inventario_actual, cantidad_pedida] = retorno_esperado

    return retornos

In [15]:
retornos_inventario = construir_retornos_inventario(costo_escasez)

print("Retornos esperados r(inventario, cantidad_pedida):")
print(np.round(retornos_inventario, 2))

Retornos esperados r(inventario, cantidad_pedida):
[[   -28.5     -5.8     15.      13.9     -1.8    -24.8]
 [    29.2     30.      28.9     13.2     -9.8 -10000. ]
 [    65.      43.9     28.2      5.2 -10000.  -10000. ]
 [    78.9     43.2     20.2 -10000.  -10000.  -10000. ]
 [    78.2     35.2 -10000.  -10000.  -10000.  -10000. ]
 [    70.2 -10000.  -10000.  -10000.  -10000.  -10000. ]]


### 3.3 Matrices de transición

Las matrices de transición **no dependen** de $(K,h,b,c)$: solo dependen de cuánto se pide y de la demanda, así que la función es igual a como la teníamos antes. Cada acción produce una matriz; las acciones infactibles reciben una transición ficticia de autorretorno (la penalización en los retornos ya impide que sean elegidas).

In [16]:
def construir_matriz_transicion_inventario(cantidad_pedida):

    matriz = np.zeros((numero_estados_inventario, numero_estados_inventario))

    for inventario_actual in range(numero_estados_inventario):

        inventario_disponible = inventario_actual + cantidad_pedida

        if inventario_disponible > capacidad_inventario:

            matriz[inventario_actual, inventario_actual] = 1.0

        else:

            for indice_demanda in range(len(valores_demanda)):

                demanda = valores_demanda[indice_demanda]
                probabilidad_demanda = probabilidades_demanda[indice_demanda]

                inventario_siguiente = max(0, inventario_disponible - demanda)

                matriz[inventario_actual, inventario_siguiente] = (
                    matriz[inventario_actual, inventario_siguiente] + probabilidad_demanda
                )

    return matriz


matrices_inventario_por_accion = []
for cantidad_pedida in range(numero_acciones_inventario):
    matrices_inventario_por_accion.append(construir_matriz_transicion_inventario(cantidad_pedida))

# Verificacion barata: cada matriz debe ser estocastica por filas
for cantidad_pedida in range(numero_acciones_inventario):
    sumas = matrices_inventario_por_accion[cantidad_pedida].sum(axis=1)
    print("Accion:", acciones_inventario[cantidad_pedida], "-> sumas de fila:", np.round(sumas, 4))

Accion: 0 -> sumas de fila: [1. 1. 1. 1. 1. 1.]
Accion: 1 -> sumas de fila: [1. 1. 1. 1. 1. 1.]
Accion: 2 -> sumas de fila: [1. 1. 1. 1. 1. 1.]
Accion: 3 -> sumas de fila: [1. 1. 1. 1. 1. 1.]
Accion: 4 -> sumas de fila: [1. 1. 1. 1. 1. 1.]
Accion: 5 -> sumas de fila: [1. 1. 1. 1. 1. 1.]


### 3.4 Solución con `jmarkov` (`dtsdp`)

Con los retornos y las matrices de transición ya construidos, resolvemos directamente con `dtsdp`: le entregamos un diccionario anidado `{periodo: {accion: matriz}}` y un arreglo 3D de retornos `(periodos, estados, acciones)`, y `solve()` hace la recursión hacia atrás por nosotros.

In [17]:
probabilidades_inventario_jmarkov = {}
for epoca in range(numero_epocas_inventario):
    numero_epoca = epocas_inventario[epoca]
    probabilidades_inventario_jmarkov[numero_epoca] = {}
    for cantidad_pedida in range(numero_acciones_inventario):
        nombre_accion = acciones_inventario[cantidad_pedida]
        probabilidades_inventario_jmarkov[numero_epoca][nombre_accion] = matrices_inventario_por_accion[cantidad_pedida]

retornos_inventario_jmarkov = np.zeros((numero_epocas_inventario, numero_estados_inventario, numero_acciones_inventario))
for epoca in range(numero_epocas_inventario):
    retornos_inventario_jmarkov[epoca, :, :] = retornos_inventario

sdp_inventario = dtsdp(
    epocas_inventario,
    estados_inventario,
    acciones_inventario,
    probabilidades_inventario_jmarkov,
    retornos_inventario_jmarkov,
    1.0
)

# F: valor optimo, forma (estados, periodos). D: cantidad optima a pedir, forma (estados, periodos)
F_inventario, D_inventario = sdp_inventario.solve(minimize=False)

print("Politica optima: unidades a pedir por inventario inicial y semana")
for inventario_actual in range(numero_estados_inventario):
    print()
    print("Inventario inicial:", inventario_actual)
    for epoca in range(numero_epocas_inventario):
        cantidad_pedida = int(D_inventario[inventario_actual, epoca])
        print("  Semana", epocas_inventario[epoca], "-> pedir", cantidad_pedida)

print()
print("Ganancia esperada desde inventario 0 en la semana 1:", round(float(F_inventario[0, 0]), 2))

Politica optima: unidades a pedir por inventario inicial y semana

Inventario inicial: 0
  Semana 1 -> pedir 4
  Semana 2 -> pedir 4
  Semana 3 -> pedir 2

Inventario inicial: 1
  Semana 1 -> pedir 3
  Semana 2 -> pedir 3
  Semana 3 -> pedir 1

Inventario inicial: 2
  Semana 1 -> pedir 0
  Semana 2 -> pedir 0
  Semana 3 -> pedir 0

Inventario inicial: 3
  Semana 1 -> pedir 0
  Semana 2 -> pedir 0
  Semana 3 -> pedir 0

Inventario inicial: 4
  Semana 1 -> pedir 0
  Semana 2 -> pedir 0
  Semana 3 -> pedir 0

Inventario inicial: 5
  Semana 1 -> pedir 0
  Semana 2 -> pedir 0
  Semana 3 -> pedir 0

Ganancia esperada desde inventario 0 en la semana 1: 94.66


### 3.5 ¿Tiene estructura $(s_t,S_t)$?

Para cada época comprobamos:

1. hasta qué inventario se ordena;
2. a qué nivel queda el inventario después de ordenar;
3. si todos los estados que ordenan llegan al mismo nivel objetivo.

In [18]:
print("Revision de la estructura (s_t, S_t):")

for epoca in range(numero_epocas_inventario):

    punto_reorden = -1
    nivel_objetivo = -1
    estructura_sS = True

    for inventario_actual in range(numero_estados_inventario):

        cantidad_pedida = int(D_inventario[inventario_actual, epoca])

        if cantidad_pedida > 0:

            punto_reorden = inventario_actual
            inventario_despues_pedido = inventario_actual + cantidad_pedida

            if nivel_objetivo == -1:
                nivel_objetivo = inventario_despues_pedido
            elif inventario_despues_pedido != nivel_objetivo:
                estructura_sS = False

    print()
    print("Semana:", epocas_inventario[epoca])
    print("  s_t (ultimo inventario que SI ordena):", punto_reorden)
    print("  S_t (nivel objetivo tras ordenar):", nivel_objetivo)
    print("  ¿Estructura (s_t,S_t) consistente?:", estructura_sS)

Revision de la estructura (s_t, S_t):

Semana: 1
  s_t (ultimo inventario que SI ordena): 1
  S_t (nivel objetivo tras ordenar): 4
  ¿Estructura (s_t,S_t) consistente?: True

Semana: 2
  s_t (ultimo inventario que SI ordena): 1
  S_t (nivel objetivo tras ordenar): 4
  ¿Estructura (s_t,S_t) consistente?: True

Semana: 3
  s_t (ultimo inventario que SI ordena): 1
  S_t (nivel objetivo tras ordenar): 2
  ¿Estructura (s_t,S_t) consistente?: True


---
## Ejercicio abierto — Inventario de medicamentos en una clínica

La clínica administra un medicamento durante cuatro semanas, con la misma estructura de costos $(K,h,b,c)$ del Ejercicio 3.

| Parámetro | Símbolo | Valor |
|:---|:---:|---:|
| Precio por unidad suministrada | $p$ | 200 |
| Costo fijo de ordenar | $K$ | 100 |
| Costo variable por unidad pedida | $c$ | 80 |
| Costo de almacenamiento | $h$ | 20 |
| Costo de escasez | $b$ | 150 |
| Capacidad máxima | | 4 |
| Horizonte | | 4 semanas |
| Descuento base | $\gamma$ | 1.0 |

La demanda sigue una Poisson con $\lambda=2$, truncada en $\{0,1,2,3,4\}$: la cola de la distribución (todo lo que sería $D\ge 4$) se acumula en $D=4$.

### Checkpoints

1. Construya la distribución truncada.
2. Construya una matriz de retornos bidimensional $r(i,a)$ usando la fórmula de la Sección 3.2, con este $K$ y este $c$.
3. Construya una lista de cinco matrices de transición (una por cada cantidad a pedir, $0$ a $4$).
4. Calcule la última época sin continuación: $V_T(i) = \max_a r(i,a)$.
5. USe `dtsdp` para las cuatro semanas.
6. Compare $\gamma=1.0$ y $\gamma=0.85$: ¿cambia la política? ¿cambia solo el valor?

### Uso crítico de IA

La IA puede ayudar a completar el ejercicio, pero el código entregado debe conservar el estilo del curso:

- ciclos explícitos, sin `zip` ni `enumerate`;
- nombres descriptivos;
- sin comprensiones de listas o diccionarios;
- `V[epoca, estado]` y `politica[epoca, estado]`;
- Usar `jmarkov`. (o la alternativa 'a mano')

Antes de aceptar cualquier solución (propia o sugerida por una IA), compare su última época con la referencia calculada en la siguiente celda: si no coincide, el error está en la construcción de retornos, no en la recursión.

In [19]:
from scipy.stats import poisson

lambda_demanda_clinica = 2

valores_demanda_clinica = np.array([0, 1, 2, 3, 4])

probabilidades_demanda_clinica = np.zeros(5)
for demanda in range(4):
    probabilidades_demanda_clinica[demanda] = poisson.pmf(demanda, lambda_demanda_clinica)

# La cola D>=4 se acumula en D=4, para que la distribucion siga sumando 1
suma_primeras_probabilidades = 0.0
for demanda in range(4):
    suma_primeras_probabilidades = suma_primeras_probabilidades + probabilidades_demanda_clinica[demanda]

probabilidades_demanda_clinica[4] = 1.0 - suma_primeras_probabilidades

print("Distribucion truncada:", np.round(probabilidades_demanda_clinica, 4))
print("Suma:", round(probabilidades_demanda_clinica.sum(), 6))

Distribucion truncada: [0.1353 0.2707 0.2707 0.1804 0.1429]
Suma: 1.0


In [20]:
capacidad_clinica = 4
precio_clinica = 200
K_clinica = 100
c_clinica = 80
costo_almacenamiento_clinica = 20
costo_escasez_clinica = 150

numero_estados_clinica = 5
numero_acciones_clinica = 5

retornos_clinica = np.zeros((numero_estados_clinica, numero_acciones_clinica))

for inventario_actual in range(numero_estados_clinica):

    for cantidad_pedida in range(numero_acciones_clinica):

        inventario_disponible = inventario_actual + cantidad_pedida

        if inventario_disponible > capacidad_clinica:

            retornos_clinica[inventario_actual, cantidad_pedida] = -100000

        else:

            if cantidad_pedida > 0:
                costo_ordenar = K_clinica + c_clinica * cantidad_pedida
            else:
                costo_ordenar = 0.0

            retorno_esperado = -costo_ordenar

            for indice_demanda in range(5):

                demanda = valores_demanda_clinica[indice_demanda]
                probabilidad = probabilidades_demanda_clinica[indice_demanda]

                ventas = min(inventario_disponible, demanda)
                sobrante = max(0, inventario_disponible - demanda)
                escasez = max(0, demanda - inventario_disponible)

                retorno_escenario = (
                    precio_clinica * ventas
                    - costo_almacenamiento_clinica * sobrante
                    - costo_escasez_clinica * escasez
                )

                retorno_esperado = retorno_esperado + probabilidad * retorno_escenario

            retornos_clinica[inventario_actual, cantidad_pedida] = retorno_esperado

valor_ultima_epoca_referencia = np.zeros(numero_estados_clinica)
for inventario_actual in range(numero_estados_clinica):
    valor_ultima_epoca_referencia[inventario_actual] = np.max(retornos_clinica[inventario_actual, :])

print("Referencia de la ultima epoca (V_T(i) = max_a r(i,a)):")
print(np.round(valor_ultima_epoca_referencia, 2))

Referencia de la ultima epoca (V_T(i) = max_a r(i,a)):
[-29.4   50.6  210.97 310.6  343.47]


### Prompt sugerido

```text
Ayúdame a completar un MDP de inventario de cuatro semanas en Python.

Usa el mismo estilo del laboratorio:
- ciclos for explícitos;
- nombres descriptivos;
- sin zip ni enumerate;
- sin comprensiones de listas o diccionarios;
- V[época, estado] y política[época, estado];
- separa el costo de ordenar en un costo fijo K y uno variable c;
- jmarkov (dtsdp) solamente al final, para verificar.

La demanda es Poisson(2), truncada en 0,1,2,3,4, acumulando la cola en 4.
Los estados son inventarios 0 a 4 y las acciones son pedidos 0 a 4.
La capacidad es 4.
Precio=200, K=100, c=80, almacenamiento=20 y escasez=150.

Primero construye una matriz bidimensional de retornos y una lista de cinco
matrices de transición. Después reutiliza una función de backward induction
para cuatro épocas. Compara gamma=1.0 y gamma=0.85. Muestra la política como
unidades a pedir por estado y época. Finalmente prepara una verificación con
dtsdp, pero no lo uses para resolver el algoritmo principal.
```

In [21]:
# Complete aqui la construccion de las matrices de transicion,
# ejecute backward_induction para cuatro epocas y compare
# gamma = 1.0 con gamma = 0.85.
#
# Antes de aceptar los resultados, compruebe que la ultima fila
# temporal de V coincide con valor_ultima_epoca_referencia.

---
<small>Universidad de los Andes · Departamento de Ingeniería Industrial · Modelos de Decisión en el Tiempo</small>